# Deploy Sample A2A Agents

## Overview

This notebook deploys a set of **sample e-commerce A2A agents** that handle stateful, multi-step operations requiring reasoning across tool calls. Each agent runs as an Amazon Bedrock AgentCore Runtime and exposes its capability via the A2A JSON-RPC protocol.

## What you will build

Deploy **3 A2A agents** using Amazon Bedrock AgentCore Runtime.

By the end you will have:
- **3 A2A agents** running on AgentCore Runtime, each backed by a Strands agent
- A **shared sample database** (DynamoDB) seeded with orders, payments, and inventory
- End-to-end tests that exercise every agent via `invoke_agent_runtime`

For MCP tools, see `00_deploy_sample_mcp_tools.ipynb`.

## Architecture

```
A2A Agents on Runtime (3)
──────────────────────────────────────────────────────────
AgentCore Runtime (A2A / HTTP JSON-RPC)
  │
  ├─ Agent: PaymentRefundAgent
  │    Tools: get_order_payment_info → issue_refund → get_refund_status
  │    Tables: orders, payments, refunds
  │
  ├─ Agent: InventoryReserveAgent
  │    Tools: reserve_inventory → get_reservation_status / release_reservation
  │    Tables: inventory, reservations
  │
  └─ Agent: ShippingUpdateAgent
       Tools: assign_carrier → create_shipment → update_shipment_status
       Tables: orders, shipments

Shared: DynamoDB (6 tables) ─── seeded with orders, payments, inventory
Invocation: invoke_agent_runtime (A2A JSON-RPC message/send)
```

## Why A2A?

Each agent handles operations that require **multi-step reasoning** across dependent tool calls — the result of one step determines what to do next:

- **PaymentRefundAgent**: validate order and payment → decide partial vs full → issue → confirm
- **InventoryReserveAgent**: check available stock → reserve → support rollback if downstream steps fail
- **ShippingUpdateAgent**: pick carrier based on weight and destination → create shipment → confirm status

## Prerequisites
- AWS account with Bedrock model access enabled (`us-west-2`)
- IAM permissions: DynamoDB, IAM, Bedrock AgentCore, ECR, CodeBuild
- Python 3.10+

---
## Agents Overview

| Agent | A2A Tool Name | Tools | DynamoDB Tables |
|---|---|---|---|
| `PaymentRefundAgent` | `payment_refund_tool` | get_order_payment_info, issue_refund, get_refund_status | orders, payments, refunds |
| `InventoryReserveAgent` | `inventory_reserve_tool` | reserve_inventory, get_reservation_status, release_reservation | inventory, reservations |
| `ShippingUpdateAgent` | `shipping_update_tool` | assign_carrier, create_shipment, update_shipment_status | orders, shipments |

---
## Sections
1. **Setup** — clients and config
2. **DynamoDB** — create tables or point to existing
3. **Deploy** — package and launch each A2A agent
4. **Test** — invoke each agent end-to-end
5. **Cleanup**

---
## 1. Setup

In [ ]:
!pip install strands-agents "strands-agents[a2a]" bedrock-agentcore bedrock-agentcore-starter-toolkit fastapi uvicorn boto3 -q

In [ ]:
import boto3, json, time, os, sys, pathlib, shutil, uuid
from datetime import datetime


# ── Config ─────────────────────────────────────────────────────────────────
# Set AWS credentials if not using Amazon SageMaker notebook
#AWS_PROFILE = "configured-aws-profile"

AWS_REGION  =  "us-west-2"
MODEL_ID    = "us.anthropic.claude-sonnet-4-20250514-v1:0"

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

session          = boto3.Session(region_name=AWS_REGION)
iam_client       = session.client("iam")
agentcore_client = session.client("bedrock-agentcore-control")
ac_data_client   = session.client("bedrock-agentcore")
logs_client      = session.client("logs")
ACCOUNT_ID       = session.client("sts").get_caller_identity()["Account"]

print(f"Account     : {ACCOUNT_ID}")
print(f"Region      : {AWS_REGION}")
print(f"Timestamp   : {timestamp}")

---
## 2. DynamoDB Table Creation


In [ ]:
# ── Auto-detect table prefix from MCP tools deployment config ────────────
import pathlib
mcp_config_path = pathlib.Path("utils/mcp_tools_config.json")
if mcp_config_path.exists():
    _mcp_cfg = json.loads(mcp_config_path.read_text())
    EXISTING_TABLE_PREFIX = f"tools-{_mcp_cfg['timestamp']}-"
    print(f"Auto-detected from MCP config: {EXISTING_TABLE_PREFIX}")
else:
    EXISTING_TABLE_PREFIX = ""
    print("No MCP config found — run 00_deploy_sample_mcp_tools first, or set EXISTING_TABLE_PREFIX manually.")

if EXISTING_TABLE_PREFIX:
    TABLE_PREFIX = EXISTING_TABLE_PREFIX
    ddb_client   = session.client("dynamodb", region_name=AWS_REGION)
    # Tables used by A2A agents: orders, payments, refunds, inventory, reservations, shipments
    TABLE_NAMES  = {
        "orders":       TABLE_PREFIX + "orders",
        "payments":     TABLE_PREFIX + "payments",
        "refunds":      TABLE_PREFIX + "refunds",
        "inventory":    TABLE_PREFIX + "inventory",
        "reservations": TABLE_PREFIX + "reservations",
        "shipments":    TABLE_PREFIX + "shipments",
    }
    # Create A2A-only tables (refunds, reservations) if they don't exist yet
    A2A_ONLY_TABLES = {
        "refunds":      "refund_id",
        "reservations": "reservation_id",
    }
    for logical, pk in A2A_ONLY_TABLES.items():
        tname = TABLE_NAMES[logical]
        try:
            ddb_client.create_table(
                TableName=tname,
                BillingMode="PAY_PER_REQUEST",
                KeySchema=[{"AttributeName": pk, "KeyType": "HASH"}],
                AttributeDefinitions=[{"AttributeName": pk, "AttributeType": "S"}],
            )
            print(f"  Created: {tname}")
        except ddb_client.exceptions.ResourceInUseException:
            print(f"  Already exists: {tname}")
    # Wait for any newly created tables
    for logical in A2A_ONLY_TABLES:
        ddb_client.get_waiter("table_exists").wait(TableName=TABLE_NAMES[logical])

    print("\nReusing existing tables:")
    for k, v in TABLE_NAMES.items():
        print(f"  {k:15s} → {v}")
else:
    print("EXISTING_TABLE_PREFIX is empty — run the cell below to create fresh tables.")

---
## 3. Deploy A2A Agents

Each cell is self-contained — deploy agents independently or run all three in sequence.

In [ ]:
# ── Deploy: payment_refund_a2a ─────────────────────────────────────────────
from bedrock_agentcore_starter_toolkit import Runtime

_template = pathlib.Path("utils/agents/payment_refund_agent.py").read_text()
_env_vars = (
    'os.environ.setdefault("ORDERS_TABLE",   "{orders}")\n'
    'os.environ.setdefault("PAYMENTS_TABLE", "{payments}")\n'
    'os.environ.setdefault("REFUNDS_TABLE",  "{refunds}")\n'
).format(**TABLE_NAMES)
pathlib.Path("payment_refund_agent.py").write_text(
    _template.replace("# TABLE_NAMES_PLACEHOLDER\n", _env_vars)
)
shutil.copy("utils/db.py", "db.py")
pathlib.Path("payment_refund_requirements.txt").write_text(
    "strands-agents[a2a]\nfastapi\nuvicorn\nboto3\n"
)
for stale in [".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"]:
    p = pathlib.Path(stale)
    if p.exists(): p.unlink()

pmt_runtime = Runtime()
pmt_runtime.configure(
    entrypoint="payment_refund_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="payment_refund_requirements.txt",
    region=AWS_REGION,
    agent_name=f"payment_refund_a2a_{timestamp}",
    protocol="A2A",
)
pmt_launch    = pmt_runtime.launch(auto_update_on_conflict=True)
pmt_agent_id  = pmt_launch.agent_id
pmt_agent_arn = pmt_launch.agent_arn
print(f"Agent ID  : {pmt_agent_id}")
print(f"Agent ARN : {pmt_agent_arn}")

from utils.iam_helpers import attach_dynamodb_policy_and_wait
attach_dynamodb_policy_and_wait(
    agentcore_client, iam_client, pmt_agent_id, "payment_refund_a2a",
    AWS_REGION, ACCOUNT_ID, TABLE_PREFIX
)

In [ ]:
# ── Deploy: inventory_reserve_a2a ──────────────────────────────────────────
from bedrock_agentcore_starter_toolkit import Runtime

_template = pathlib.Path("utils/agents/inventory_reserve_agent.py").read_text()
_env_vars = (
    'os.environ.setdefault("INVENTORY_TABLE",    "{inventory}")\n'
    'os.environ.setdefault("RESERVATIONS_TABLE", "{reservations}")\n'
).format(**TABLE_NAMES)
pathlib.Path("inventory_reserve_agent.py").write_text(
    _template.replace("# TABLE_NAMES_PLACEHOLDER\n", _env_vars)
)
shutil.copy("utils/db.py", "db.py")
pathlib.Path("inventory_reserve_requirements.txt").write_text(
    "strands-agents[a2a]\nfastapi\nuvicorn\nboto3\n"
)
for stale in [".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"]:
    p = pathlib.Path(stale)
    if p.exists(): p.unlink()

inv_runtime = Runtime()
inv_runtime.configure(
    entrypoint="inventory_reserve_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="inventory_reserve_requirements.txt",
    region=AWS_REGION,
    agent_name=f"inventory_reserve_a2a_{timestamp}",
    protocol="A2A",
)
inv_launch    = inv_runtime.launch(auto_update_on_conflict=True)
inv_agent_id  = inv_launch.agent_id
inv_agent_arn = inv_launch.agent_arn
print(f"Agent ID  : {inv_agent_id}")
print(f"Agent ARN : {inv_agent_arn}")

from utils.iam_helpers import attach_dynamodb_policy_and_wait
attach_dynamodb_policy_and_wait(
    agentcore_client, iam_client, inv_agent_id, "inventory_reserve_a2a",
    AWS_REGION, ACCOUNT_ID, TABLE_PREFIX
)

In [ ]:
# ── Deploy: shipping_update_a2a ────────────────────────────────────────────
from bedrock_agentcore_starter_toolkit import Runtime

_template = pathlib.Path("utils/agents/shipping_update_agent.py").read_text()
_env_vars = (
    'os.environ.setdefault("ORDERS_TABLE",    "{orders}")\n'
    'os.environ.setdefault("SHIPMENTS_TABLE", "{shipments}")\n'
).format(**TABLE_NAMES)
pathlib.Path("shipping_update_agent.py").write_text(
    _template.replace("# TABLE_NAMES_PLACEHOLDER\n", _env_vars)
)
shutil.copy("utils/db.py", "db.py")
pathlib.Path("shipping_update_requirements.txt").write_text(
    "strands-agents[a2a]\nfastapi\nuvicorn\nboto3\n"
)
for stale in [".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"]:
    p = pathlib.Path(stale)
    if p.exists(): p.unlink()

ship_runtime = Runtime()
ship_runtime.configure(
    entrypoint="shipping_update_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="shipping_update_requirements.txt",
    region=AWS_REGION,
    agent_name=f"shipping_update_a2a_{timestamp}",
    protocol="A2A",
)
ship_launch    = ship_runtime.launch(auto_update_on_conflict=True)
ship_agent_id  = ship_launch.agent_id
ship_agent_arn = ship_launch.agent_arn
print(f"Agent ID  : {ship_agent_id}")
print(f"Agent ARN : {ship_agent_arn}")

from utils.iam_helpers import attach_dynamodb_policy_and_wait
attach_dynamodb_policy_and_wait(
    agentcore_client, iam_client, ship_agent_id, "shipping_update_a2a",
    AWS_REGION, ACCOUNT_ID, TABLE_PREFIX
)

### Save deployment config for downstream notebooks

Writes `util/a2a_agents_config.json` so that `07_planner_executor_live.ipynb` can load
the A2A agent ARNs and descriptions without manual copy-paste.

In [ ]:
import pathlib

a2a_config = {
    "timestamp": timestamp,
    "agents": {
        "payment_refund": {
            "agent_id": pmt_agent_id,
            "agent_arn": pmt_agent_arn,
            "name": "PaymentRefundAgent",
            "description": "Issues refunds with multi-step validation: verify order/payment, issue refund, confirm status.",
            "skills": ["get_order_payment_info", "issue_refund", "get_refund_status"]
        },
        "inventory_reserve": {
            "agent_id": inv_agent_id,
            "agent_arn": inv_agent_arn,
            "name": "InventoryReserveAgent",
            "description": "Reserves inventory for orders to prevent overselling, with rollback support.",
            "skills": ["reserve_inventory", "get_reservation_status", "release_reservation"]
        },
        "shipping_update": {
            "agent_id": ship_agent_id,
            "agent_arn": ship_agent_arn,
            "name": "ShippingUpdateAgent",
            "description": "Assigns carriers, creates shipments, and updates shipment status for orders.",
            "skills": ["assign_carrier", "create_shipment", "update_shipment_status"]
        }
    }
}

config_path = pathlib.Path("utils/a2a_agents_config.json")
config_path.write_text(json.dumps(a2a_config, indent=2))
print(f"Saved: {config_path}")

---
## 4. Test

The `a2a_call` helper matches the `bedrock-agentcore-starter-toolkit` invocation pattern:
- Passes `runtimeSessionId`
- Injects `Accept: text/event-stream, application/json` via boto3 event hook
- Handles both streaming and non-streaming EventStream responses

In [ ]:
def a2a_call(agent_arn, prompt, verbose=False):
    """Invoke an A2A agent via invoke_agent_runtime using message/send."""
    session_id  = str(uuid.uuid4())
    message_id  = str(uuid.uuid4())
    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "messageId": message_id,
                "parts": [{"kind": "text", "text": prompt}]
            }
        }
    })
    if verbose:
        print(f"[a2a_call] ARN     : {agent_arn}")
        print(f"[a2a_call] session : {session_id}")
        print(f"[a2a_call] payload : {payload}")

    def _add_accept(request, **kwargs):
        request.headers.add_header("Accept", "text/event-stream, application/json")

    ac_data_client.meta.events.register_first(
        "before-sign.bedrock-agentcore.InvokeAgentRuntime", _add_accept
    )
    try:
        resp = ac_data_client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn,
            qualifier="DEFAULT",
            runtimeSessionId=session_id,
            contentType="application/json",
            payload=payload,
        )
    finally:
        ac_data_client.meta.events.unregister(
            "before-sign.bedrock-agentcore.InvokeAgentRuntime", _add_accept
        )

    ct   = resp.get("contentType", "")
    body = resp["response"]

    if "text/event-stream" in ct:
        texts = []
        for line in body.iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8") if isinstance(line, bytes) else line
                if verbose: print(f"[stream] {line}")
                if line.startswith("data: "):
                    try:
                        chunk = json.loads(line[6:])
                        texts.append(chunk if isinstance(chunk, str) else json.dumps(chunk))
                    except Exception:
                        texts.append(line[6:])
        return "\n".join(texts) or "(empty streaming response)"

    # Non-streaming: collect EventStream chunks
    chunks = []
    for event in body:
        chunks.append(event.decode("utf-8") if isinstance(event, bytes) else str(event))
    raw = "".join(chunks)
    if verbose: print(f"[raw] {raw}")
    try:
        data = json.loads(raw)
        # message/send response: result.parts[]
        parts = data.get("result", {}).get("parts", [])
        if parts:
            return "\n".join(p.get("text", "") for p in parts if p.get("kind") == "text" or p.get("type") == "text")
        # fallback: result.status.message.parts (older spec)
        parts = (data.get("result", {}).get("status", {}).get("message", {}).get("parts", []))
        if parts:
            return "\n".join(p.get("text", "") for p in parts if p.get("type") == "text")
        return json.dumps(data, indent=2)
    except Exception:
        return raw

print("a2a_call helper ready (method: message/send).")

In [ ]:
# ── Test: payment_refund_a2a ───────────────────────────────────────────────
print("=" * 60); print("payment_refund_tool (A2A)"); print("=" * 60)
print(a2a_call(pmt_agent_arn,
    "Issue a full refund of $59.98 for order ORD-1001 due to customer request, "
    "then confirm the refund status."))

In [ ]:
# ── Test: inventory_reserve_a2a ────────────────────────────────────────────
print("=" * 60); print("inventory_reserve_tool (A2A)"); print("=" * 60)
print(a2a_call(inv_agent_arn,
    "Reserve 5 units of WIDGET-42 for order ORD-1004, then confirm the reservation status."))

In [ ]:
# ── Test: shipping_update_a2a ──────────────────────────────────────────────
print("=" * 60); print("shipping_update_tool (A2A)"); print("=" * 60)
print(a2a_call(ship_agent_arn,
    "Create a shipment for order ORD-1001. Package weighs 2kg, ships to WA. Pick the best carrier."))

---
## 5. Cleanup (Optional) - PLEASE DO NOT EXECUTE THIS CELL UNTILL YOU HAVE EXECUTED PLANNER EXECUTOR NOTEBOOK

In [ ]:
# ── Cleanup: delete A2A runtimes, IAM roles, DynamoDB tables ──────────────
# Set CLEANUP_TIMESTAMP if cleaning up a different deployment.
CLEANUP_TIMESTAMP = ""   # leave empty to use current session's timestamp

_ts = CLEANUP_TIMESTAMP or timestamp
print(f"Cleaning up deployment: {_ts}\n")

# 1. Delete A2A runtimes
print("Deleting A2A runtimes...")
for rt in agentcore_client.list_agent_runtimes().get("agentRuntimes", []):
    if _ts in rt.get("agentRuntimeName", ""):
        try:
            agentcore_client.delete_agent_runtime(agentRuntimeId=rt["agentRuntimeId"])
            print(f"  Deleted: {rt['agentRuntimeName']}")
        except Exception as e:
            print(f"  {rt['agentRuntimeId']}: {e}")

# 2. Delete IAM roles (execution roles created by the toolkit)
print("\nDeleting IAM roles...")
try:
    paginator = iam_client.get_paginator("list_roles")
    for page in paginator.paginate():
        for role in page["Roles"]:
            rname = role["RoleName"]
            if _ts not in rname:
                continue
            try:
                for p in iam_client.list_role_policies(RoleName=rname)["PolicyNames"]:
                    iam_client.delete_role_policy(RoleName=rname, PolicyName=p)
                for p in iam_client.list_attached_role_policies(RoleName=rname)["AttachedPolicies"]:
                    iam_client.detach_role_policy(RoleName=rname, PolicyArn=p["PolicyArn"])
                iam_client.delete_role(RoleName=rname)
                print(f"  Deleted: {rname}")
            except Exception as e:
                print(f"  {rname}: {e}")
except Exception as e:
    print(f"  IAM scan failed: {e}")

# 3. Delete DynamoDB tables
print("\nDeleting DynamoDB tables...")
_prefix = f"tools-{_ts}-"
for tname in ddb_client.list_tables().get("TableNames", []):
    if tname.startswith(_prefix):
        try:
            ddb_client.delete_table(TableName=tname)
            print(f"  Deleted: {tname}")
        except Exception as e:
            print(f"  {tname}: {e}")

# 4. Remove local temp files
print("\nRemoving local temp files...")
for f in ["payment_refund_agent.py",    "payment_refund_requirements.txt",
          "inventory_reserve_agent.py", "inventory_reserve_requirements.txt",
          "shipping_update_agent.py",   "shipping_update_requirements.txt",
          "db.py", ".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"]:
    p = pathlib.Path(f)
    if p.exists():
        p.unlink()
        print(f"  Removed: {f}")

# 5. Config file
_cfg = pathlib.Path("utils/a2a_agents_config.json")
if _cfg.exists():
    _cfg.unlink()
    print(f"  Removed: {_cfg}")

print("\nCleanup complete.")